<a href="https://colab.research.google.com/github/PeroronShine/education_fefu_2/blob/main/cubernetic/%D0%BA%D0%B8%D0%B1%D0%B5%D1%80%D0%BD%D0%B5%D1%82%D0%B8%D0%BA%D0%B04_%D0%BA%D0%BE%D0%B4%D0%A5%D1%8D%D0%BC%D0%BC%D0%B8%D0%BD%D0%B3%D0%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
def calculate_parity_bits_length(m):
    r = 0
    while (1 << r) < m + r + 1:
        r += 1
    return r

def encode_hamming_block(data_bits, verbose=True):
    m = len(data_bits)
    r = calculate_parity_bits_length(m)
    n = m + r

    if verbose:
        print(f"🔹 Кодирование блока данных: {data_bits} → длина данных = {m}")
        print(f"🔹 Требуется r = {r} контрольных битов → общий размер блока n = {n}")

    # Инициализируем закодированный блок
    encoded = [0] * n
    j = 0
    for i in range(n):
        if (i + 1) & i != 0:  # не степень двойки → информационный бит
            encoded[i] = data_bits[j]
            j += 1

    if verbose:
        print(f"\n🔹 После размещения информационных битов (позиции 3,5,6,7...):")
        print(f"   Позиции (1-based): {[i+1 for i in range(n)]}")
        print(f"   Блок (временно):    {encoded}")
        print()

    # Вычисляем каждый контрольный бит
    for i in range(r):
        parity_pos = (1 << i) - 1
        parity_bit_name = f"P{1 << i}"
        parity = 0
        participating_bits = []  # Список (позиция, значение)

        for j in range(n):
            pos = j + 1
            if pos & (1 << i):  # если i-й бит в номере позиции установлен
                participating_bits.append((pos, encoded[j]))
                parity ^= encoded[j]

        # Присваиваем вычисленное значение
        encoded[parity_pos] = parity

        if verbose:
            pos_list = [f"{pos}({val})" for pos, val in participating_bits]
            print(f"🧮 Вычисление {parity_bit_name} (контрольный бит в позиции {parity_pos + 1}):")
            print(f"   Участвуют биты в позициях: {', '.join(pos_list)}")
            print(f"   XOR = {' ^ '.join(str(val) for _, val in participating_bits)} = {parity}")
            print(f"   → {parity_bit_name} = {parity}")
            print()

    if verbose:
        print(f"✅ Окончательный закодированный блок: {encoded}")
        print("-" * 50)

    return encoded

def decode_hamming_block(received, verbose=True):
    n = len(received)
    r = 0
    while (1 << r) < n + 1:
        r += 1
    m = n - r

    if verbose:
        print(f"🔹 Декодирование блока длины {n}: {received}")
        print(f"🔹 Определяем: контрольных битов r = {r}, информационных m = {m}")
        print(f"   Позиции: {[i+1 for i in range(n)]}")
        print()

    # Вычисляем где ошибка: проверяем каждый контрольный бит
    syndrome = 0
    syndrome_bits_info = []  # для наглядного вывода

    for i in range(r):
        parity_bit_power = 1 << i  # 1, 2, 4, 8...
        parity_bit_name = f"P{parity_bit_power}"
        parity_pos = parity_bit_power - 1  # 0-based позиция контрольного бита

        parity_computed = 0
        participating_bits = []

        for j in range(n):
            pos = j + 1  # 1-based
            if pos & parity_bit_power:  # если бит участвует в проверке Pi
                participating_bits.append((pos, received[j]))
                parity_computed ^= received[j]

        # Если чётность не нулевая → ошибка в этой группе
        if parity_computed != 0:
            syndrome += parity_bit_power
            syndrome_bits_info.append((parity_bit_name, parity_computed))

        if verbose:
            pos_list = [f"{pos}({val})" for pos, val in participating_bits]
            print(f"🧮 Проверка {parity_bit_name} (должно быть чётное число единиц):")
            print(f"   Биты в позициях: {', '.join(pos_list)}")
            print(f"   XOR = {' ^ '.join(str(val) for _, val in participating_bits)} = {parity_computed}")
            if parity_computed == 0:
                print(f"   → Чётность соблюдена ✅")
            else:
                print(f"   → Нарушена чётность ❌")
            print()

    if verbose:
        if syndrome == 0:
            print("✅ 0 → ошибок не обнаружено.")
        else:
            print(f"❗ Ошибка в символе №{syndrome} ")

    # Исправление ошибки (если позиция в пределах блока)
    error_pos_1based = syndrome
    error_pos_0based = syndrome - 1
    if syndrome != 0:
        if 1 <= error_pos_1based <= n:
            if verbose:
                old_val = received[error_pos_0based]
                print(f"Исправляем бит в позиции {error_pos_1based}: {old_val} → {1 - old_val}")
            received[error_pos_0based] ^= 1
            if verbose:
                print(f"   Блок после исправления: {received}")
        else:
            if verbose:
                print(f"Ошибка указывает на позицию {error_pos_1based}, но блок имеет длину {n} → ошибка неисправима.")
    elif verbose:
        print("   Блок остаётся без изменений.")

    # Извлекаем информационные биты (все, кроме позиций 1,2,4,8,...)
    data = []
    info_positions = []
    for i in range(n):
        pos = i + 1
        if pos & (pos - 1) != 0:  # не степень двойки → информационный бит
            data.append(received[i])
            info_positions.append(pos)

    if verbose:
        print(f"\nИзвлекаем информационные биты из позиций: {info_positions}")
        print(f"   Результат: {data[:m]}")
        print("-" * 50)

    return data[:m]

def hamming_encode(message_bits, verbose=True):
    original_length = len(message_bits)
    # размер блока на которые делим исходное сообщение
    chunk_size = 4
    padded = message_bits + [0] * ((-len(message_bits)) % chunk_size)

    if verbose:
        print(f"=== КОДИРОВАНИЕ ===")
        print(f"Исходное сообщение: {message_bits} (длина = {original_length})")
        if len(padded) != original_length:
            print(f"Дополнено нулями до длины {len(padded)}: {padded}")
        else:
            print("Сообщение уже кратно 4 — дополнение не требуется.")

    encoded = []
    for idx, chunk in enumerate([padded[i:i+4] for i in range(0, len(padded), 4)]):
        if verbose:
            print(f"\n--- Блок {idx + 1} ---")
        encoded_block = encode_hamming_block(chunk, verbose=verbose)
        encoded.extend(encoded_block)

    if verbose:
        print(f"\nПолное сообщение:    {encoded}\n")
        print("-" * 50)

    return encoded, original_length

def hamming_decode(encoded_bits, original_length, verbose=True):
    block_size = 7
    decoded_bits = []

    if verbose:
        print(f"Ожидаемая длина исходного сообщения: {original_length}")

    blocks = [encoded_bits[i:i+block_size] for i in range(0, len(encoded_bits), block_size)]
    for idx, block in enumerate(blocks):
        block = (block + [0] * block_size)[:block_size]
        if verbose:
            print(f"\n--- Блок {idx + 1} ---")
        data = decode_hamming_block(block, verbose=verbose)
        decoded_bits.extend(data)

    result = decoded_bits[:original_length]
    if verbose:
        print(f"✅ Восстановленное сообщение: {result}\n")

    return result

if __name__ == "__main__":
    message = [1, 0, 1, 1]

    print("🔤 Входное сообщение (биты):", message, "\n")

    encoded, orig_len = hamming_encode(message, verbose=True)
    corrupted = encoded[:]

    # индекс символа для ошибки
    error_index = 4
    corrupted[error_index] ^= 1

    print("💥 Вносим ошибку в позицию", error_index + 1)
    print("Сообщение с ошибкой:", corrupted, "\n")

    decoded = hamming_decode(corrupted, orig_len, verbose=True)

    print("Исходное: ", message)
    print("Восстановленное:", decoded)

🔤 Входное сообщение (биты): [1, 0, 1, 1] 

=== КОДИРОВАНИЕ ===
Исходное сообщение: [1, 0, 1, 1] (длина = 4)
Сообщение уже кратно 4 — дополнение не требуется.

--- Блок 1 ---
🔹 Кодирование блока данных: [1, 0, 1, 1] → длина данных = 4
🔹 Требуется r = 3 контрольных битов → общий размер блока n = 7

🔹 После размещения информационных битов (позиции 3,5,6,7...):
   Позиции (1-based): [1, 2, 3, 4, 5, 6, 7]
   Блок (временно):    [0, 0, 1, 0, 0, 1, 1]

🧮 Вычисление P1 (контрольный бит в позиции 1):
   Участвуют биты в позициях: 1(0), 3(1), 5(0), 7(1)
   XOR = 0 ^ 1 ^ 0 ^ 1 = 0
   → P1 = 0

🧮 Вычисление P2 (контрольный бит в позиции 2):
   Участвуют биты в позициях: 2(0), 3(1), 6(1), 7(1)
   XOR = 0 ^ 1 ^ 1 ^ 1 = 1
   → P2 = 1

🧮 Вычисление P4 (контрольный бит в позиции 4):
   Участвуют биты в позициях: 4(0), 5(0), 6(1), 7(1)
   XOR = 0 ^ 0 ^ 1 ^ 1 = 0
   → P4 = 0

✅ Окончательный закодированный блок: [0, 1, 1, 0, 0, 1, 1]
--------------------------------------------------

Полное сообщение:  

In [ ]:
def encode_hamming_extended_8_4(data_bits, verbose=True):
    if len(data_bits) != 4:
        raise ValueError("Для (8,4)-кода требуется ровно 4 информационных бита.")

    d1, d2, d3, d4 = data_bits

    if verbose:
        print(f"🔹 Кодирование блока данных: {data_bits}")
        print("   Информационные биты: D1={}, D2={}, D3={}, D4={}".format(d1, d2, d3, d4))
        print("   Позиции (1-based): 1=P1, 2=P2, 3=D1, 4=P3, 5=D2, 6=D3, 7=D4, 8=P0 (общая чётность)")

    # Вычисляем контрольные биты для (7,4)-кода
    p1 = d1 ^ d2 ^ d4
    p2 = d1 ^ d3 ^ d4
    p3 = d2 ^ d3 ^ d4

    block_7 = [p1, p2, d1, p3, d2, d3, d4]

    if verbose:
        print("\n   После размещения информационных битов (позиции 3,5,6,7):")
        print("   Позиции (1-based): [1, 2, 3, 4, 5, 6, 7]")
        print("   Блок (7 бит):      {}".format(block_7))
        print()

        # Вывод вычисления P1
        bits_p1 = [(1, p1), (3, d1), (5, d2), (7, d4)]
        vals_p1 = [str(b) for _, b in bits_p1]
        print("   Вычисление P1 (позиция 1): XOR битов в позициях 1,3,5,7")
        print("     Участвующие биты: " + ", ".join(f"{pos}({val})" for pos, val in bits_p1))
        print("     XOR = {} = {}".format(" ^ ".join(vals_p1), p1))
        print("     → P1 = {}".format(p1))

        # P2
        bits_p2 = [(2, p2), (3, d1), (6, d3), (7, d4)]
        vals_p2 = [str(b) for _, b in bits_p2]
        print("   Вычисление P2 (позиция 2): XOR битов в позициях 2,3,6,7")
        print("     Участвующие биты: " + ", ".join(f"{pos}({val})" for pos, val in bits_p2))
        print("     XOR = {} = {}".format(" ^ ".join(vals_p2), p2))
        print("     → P2 = {}".format(p2))

        # P3
        bits_p3 = [(4, p3), (5, d2), (6, d3), (7, d4)]
        vals_p3 = [str(b) for _, b in bits_p3]
        print("   Вычисление P3 (позиция 4): XOR битов в позициях 4,5,6,7")
        print("     Участвующие биты: " + ", ".join(f"{pos}({val})" for pos, val in bits_p3))
        print("     XOR = {} = {}".format(" ^ ".join(vals_p3), p3))
        print("     → P3 = {}".format(p3))

    # Добавляем общий бит чётности P0 (позиция 8)
    total_xor = p1 ^ p2 ^ d1 ^ p3 ^ d2 ^ d3 ^ d4
    p0 = total_xor  # чтобы XOR всех 8 битов был 0 (чётная общая чётность)

    encoded = block_7 + [p0]

    if verbose:
        all_bits = [(i+1, encoded[i]) for i in range(8)]
        vals_all = [str(b) for _, b in all_bits]
        print("   Вычисление P0 (позиция 8, общая чётность): XOR всех 8 битов должен быть 0")
        print("     Текущий XOR первых 7 битов = {}".format(total_xor))
        print("     → P0 = {}".format(p0))
        print("     Полный блок (8 бит): {}".format(encoded))
        print("-" * 50)

    return encoded


def decode_hamming_extended_8_4(received, verbose=True):
    if len(received) < 8:
        received = (received + [0] * 8)[:8]
    else:
        received = received[:8]

    p1, p2, d1, p3, d2, d3, d4, p0 = received

    if verbose:
        print("🔹 Декодирование блока длины 8: {}".format(received))
        print("   Позиции: [1, 2, 3, 4, 5, 6, 7, 8]")

    # Проверочные уравнения (как при кодировании)
    s1 = p1 ^ d1 ^ d2 ^ d4
    s2 = p2 ^ d1 ^ d3 ^ d4
    s3 = p3 ^ d2 ^ d3 ^ d4

    syndrome = s1 + (s2 << 1) + (s3 << 2)  # от 0 до 7

    overall_parity = p1 ^ p2 ^ d1 ^ p3 ^ d2 ^ d3 ^ d4 ^ p0

    if verbose:
        print("   Проверка P1 (позиции 1,3,5,7): XOR = {} → s1 = {}".format(p1 ^ d1 ^ d2 ^ d4, s1))
        print("   Проверка P2 (позиции 2,3,6,7): XOR = {} → s2 = {}".format(p2 ^ d1 ^ d3 ^ d4, s2))
        print("   Проверка P3 (позиции 4,5,6,7): XOR = {} → s3 = {}".format(p3 ^ d2 ^ d3 ^ d4, s3))
        print("   Синдром (S3 S2 S1) = {}{}{} → {}".format(s3, s2, s1, syndrome))
        print("   Общая чётность (XOR всех 8 битов) = {}".format(overall_parity))

    corrected = received[:]

    if syndrome == 0:
        if overall_parity == 0:
            if verbose:
                print("   Ошибок не обнаружено.")
        else:
            if verbose:
                print("   Ошибка только в бите общей чётности (позиция 8). Исправляем.")
            corrected[7] ^= 1
    else:
        if overall_parity == 1:
            error_pos = syndrome  # 1-based
            if 1 <= error_pos <= 8:
                if verbose:
                    old_val = corrected[error_pos - 1]
                    print("   Обнаружена одиночная ошибка в позиции {}. Исправляем: {} → {}".format(
                        error_pos, old_val, 1 - old_val))
                corrected[error_pos - 1] ^= 1
            else:
                if verbose:
                    print("   Синдром указывает на позицию {}, выходящую за пределы блока.".format(error_pos))
        else:
            if verbose:
                print("   Обнаружено ДВЕ ошибки. Исправление невозможно.")

    # Извлекаем информационные биты: позиции 3,5,6,7 → индексы 2,4,5,6
    data = [corrected[i] for i in [2, 4, 5, 6]]

    if verbose:
        print("   Извлечённые информационные биты (позиции 3,5,6,7): {}".format(data))
        print("-" * 50)

    return data


def hamming_encode(message_bits, verbose=True):
    original_length = len(message_bits)
    chunk_size = 4
    padded = message_bits + [0] * ((-len(message_bits)) % chunk_size)

    if verbose:
        print("=== КОДИРОВАНИЕ (8,4) ===")
        print("Исходное сообщение: {} (длина = {})".format(message_bits, original_length))
        if len(padded) != original_length:
            print("Дополнено нулями до длины {}: {}".format(len(padded), padded))
        else:
            print("Сообщение уже кратно 4 — дополнение не требуется.")

    encoded = []
    blocks = [padded[i:i+4] for i in range(0, len(padded), 4)]
    for idx, chunk in enumerate(blocks):
        if verbose:
            print("\n--- Блок {} ---".format(idx + 1))
        block_encoded = encode_hamming_extended_8_4(chunk, verbose=verbose)
        encoded.extend(block_encoded)

    if verbose:
        print("\nЗакодированное сообщение: {}".format(encoded))
        print("-" * 50)

    return encoded, original_length


def hamming_decode(encoded_bits, original_length, verbose=True):
    block_size = 8
    decoded_bits = []

    if verbose:
        print("Ожидаемая исходная длина: {}".format(original_length))

    blocks = [encoded_bits[i:i+block_size] for i in range(0, len(encoded_bits), block_size)]
    for idx, block in enumerate(blocks):
        if verbose:
            print("\n--- Блок {} ---".format(idx + 1))
        data = decode_hamming_extended_8_4(block, verbose=verbose)
        decoded_bits.extend(data)

    result = decoded_bits[:original_length]
    if verbose:
        print("\nВосстановленное сообщение: {}".format(result))

    return result


if __name__ == "__main__":
    message = [1, 0, 0, 1]

    print("Входное сообщение (биты):", message, "\n")

    encoded, orig_len = hamming_encode(message, verbose=True)

    # Одиночная ошибка
    corrupted = encoded[:]
    error_index = 0
    corrupted[error_index] ^= 1
    print("Сообщение с ошибкой:     ", corrupted)

    decoded = hamming_decode(corrupted, orig_len, verbose=True)
    print("\nИсходное:     ", message)
    print("Восстановлено:", decoded)

    print("\n" + "="*60 + "\n")

    # Две ошибки
    corrupted2 = encoded[:]
    corrupted2[0] ^= 1
    corrupted2[2] ^= 1
    print("Сообщение с ошибками:", corrupted2)

    decoded2 = hamming_decode(corrupted2, orig_len, verbose=True)

Входное сообщение (биты): [1, 0, 0, 1] 

=== КОДИРОВАНИЕ (8,4) ===
Исходное сообщение: [1, 0, 0, 1] (длина = 4)
Сообщение уже кратно 4 — дополнение не требуется.

--- Блок 1 ---
🔹 Кодирование блока данных: [1, 0, 0, 1]
   Информационные биты: D1=1, D2=0, D3=0, D4=1
   Позиции (1-based): 1=P1, 2=P2, 3=D1, 4=P3, 5=D2, 6=D3, 7=D4, 8=P0 (общая чётность)

   После размещения информационных битов (позиции 3,5,6,7):
   Позиции (1-based): [1, 2, 3, 4, 5, 6, 7]
   Блок (7 бит):      [0, 0, 1, 1, 0, 0, 1]

   Вычисление P1 (позиция 1): XOR битов в позициях 1,3,5,7
     Участвующие биты: 1(0), 3(1), 5(0), 7(1)
     XOR = 0 ^ 1 ^ 0 ^ 1 = 0
     → P1 = 0
   Вычисление P2 (позиция 2): XOR битов в позициях 2,3,6,7
     Участвующие биты: 2(0), 3(1), 6(0), 7(1)
     XOR = 0 ^ 1 ^ 0 ^ 1 = 0
     → P2 = 0
   Вычисление P3 (позиция 4): XOR битов в позициях 4,5,6,7
     Участвующие биты: 4(1), 5(0), 6(0), 7(1)
     XOR = 1 ^ 0 ^ 0 ^ 1 = 1
     → P3 = 1
   Вычисление P0 (позиция 8, общая чётность): XOR всех